# SQL Refresher - Solutions

## imports

In [1]:
import math
import numpy as np
import pandas as pd

import psycopg2

## my_select_query_pandas() - function to run a select query and return rows in a Pandas dataframe

In [2]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

## Connect to the Postgres database

In [3]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

## Create a cursor for the connection

In [4]:
cursor = connection.cursor()

## You try it - select all the columns and all the rows from the zip_codes table

In [5]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select * 
from zip_codes

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,latitude,longitude,city,state,population,area,density,time_zone
0,08074,39.7158,-75.1640,Richwood,NJ,15,0.0886,169.39,America/New_York
1,08240,39.4873,-74.5318,Pomona,NJ,2293,1.5196,1508.93,America/New_York
2,08876,40.5880,-74.6874,Somerville,NJ,22059,15.1172,1459.20,America/New_York
3,10001,40.7506,-73.9972,New York,NY,22924,0.6675,34341.44,America/New_York
4,32026,30.0541,-82.1815,Raiford,FL,1907,0.6333,3011.38,America/New_York
...,...,...,...,...,...,...,...,...,...
32718,47367,40.0827,-85.3872,Oakville,IN,23,0.0866,265.47,America/Indiana/Indianapolis
32719,63079,38.2606,-91.0998,Stanton,MO,24,0.3523,68.12,America/Chicago
32720,63738,37.0893,-89.9574,Brownwood,MO,31,0.1171,264.70,America/Chicago
32721,68954,40.6227,-98.2374,Inland,NE,14,1.7437,8.03,America/Chicago


## You try it - select the following columns from the zip_codes table: zip, city, state, population, add a derived column for a 30% increase in population, and give it a meaningful alias

In [6]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select zip, city, state, population, (population * 1.3) as population_plus_30_pct
from zip_codes

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population,population_plus_30_pct
0,08074,Richwood,NJ,15,19.5
1,08240,Pomona,NJ,2293,2980.9
2,08876,Somerville,NJ,22059,28676.7
3,10001,New York,NY,22924,29801.2
4,32026,Raiford,FL,1907,2479.1
...,...,...,...,...,...
32718,47367,Oakville,IN,23,29.9
32719,63079,Stanton,MO,24,31.2
32720,63738,Brownwood,MO,31,40.3
32721,68954,Inland,NE,14,18.2


## You try it - using the zip_codes table, find top 5 zip codes with highest density, display the zip, city, state, and density

In [7]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select zip, city, state, density
from zip_codes
order by density desc
limit 5

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,density
0,20052,Washington,DC,172372.59
1,10162,New York,NY,146388.52
2,10028,New York,NY,144096.64
3,10075,New York,NY,141792.32
4,20390,Washington,DC,138322.26


## You try it - using the zip_codes table, find zip codes that are in Californa, Arizona, or Washington state with a population > 10,000, display the zip, city, state, and population with the lowest population first 

In [8]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select zip, city, state, population
from zip_codes
where state in ('CA', 'AZ', 'WA') and population > 10000
order by population

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,zip,city,state,population
0,98070,Vashon,WA,10036
1,98528,Belfair,WA,10056
2,98346,Kingston,WA,10106
3,96019,Shasta Lake,CA,10111
4,93219,Earlimart,CA,10167
...,...,...,...,...
1420,92335,Fontana,CA,99284
1421,90201,Bell Gardens,CA,102433
1422,91331,Pacoima,CA,105799
1423,90650,Norwalk,CA,105886


## You try it - using the zip_code table, find all cities, their state, and the number of zip codes in that city, in which the city starts with a C, sort by state, then by city

In [9]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select city, state, count(*) as total_zips
from zip_codes
where city like 'C%'
group by city, state
order by state, city 

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,city,state,total_zips
0,Cantwell,AK,1
1,Chalkyitsik,AK,1
2,Chefornak,AK,1
3,Chevak,AK,1
4,Chignik,AK,1
...,...,...,...
2495,Cody,WY,1
2496,Cokeville,WY,1
2497,Cora,WY,1
2498,Cowley,WY,1


## You try it - using the zip_code table, find the cities and their state, that start with C, that have more than 5 zip codes, and have a minimum population > 10000, sort by state, then by city

In [10]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select city, state, count(*) as total_zips, min(population) as min_population
from zip_codes
where city like 'C%'
group by city, state
having count(*) > 5 and min(population) > 10000
order by state, city 

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,city,state,total_zips,min_population
0,Chandler,AZ,6,33302
1,Cape Coral,FL,6,22354
2,Conroe,TX,7,13245
3,Chesapeake,VA,6,16342


## You try it - using the states table, create temp_states_1 that contains state names that start with a C or a T, create temp_states_2 that contains state names that start with a T, demonstrate set operations, drop the temp tables when you are done

In [11]:
connection.rollback()

query = """

drop table if exists temp_states_1;

drop table if exists temp_states_2;

create table temp_states_1
as
select state_name
from states
where state_name like 'C%' or state_name like 'T%'
;

create table temp_states_2
as
select state_name
from states
where state_name like 'T%'
;

"""

cursor.execute(query)

connection.commit()

In [12]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from temp_states_1
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state_name
0,California
1,Colorado
2,Connecticut
3,Tennessee
4,Texas


In [13]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from temp_states_2
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state_name
0,Tennessee
1,Texas


In [14]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_states_1)
union
(select *
from temp_states_2)
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state_name
0,California
1,Colorado
2,Connecticut
3,Tennessee
4,Texas


In [15]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_states_1)
union all
(select *
from temp_states_2)
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state_name
0,California
1,Colorado
2,Connecticut
3,Tennessee
4,Tennessee
5,Texas
6,Texas


In [16]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_states_1)
intersect
(select *
from temp_states_2)
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state_name
0,Tennessee
1,Texas


In [17]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select *
from temp_states_1)
except
(select *
from temp_states_2)
order by 1

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,state_name
0,California
1,Colorado
2,Connecticut


In [18]:
connection.rollback()

query = """

drop table if exists temp_states_1;

drop table if exists temp_states_2;

"""

cursor.execute(query)

connection.commit()

## You try it - For each store, for each day of the week, for each product, list the total number sold.  

Hints:  

extract(dow from sa.sale_date) will give you the day of week as an integer from 0 to 6 with 0 = Sunday.   You will find this helpful for ordering.

to_char(sa.sale_date, 'Day') will give you the day of week in the form "Sunday", "Monday", etc. You will find this more user friendly than the numeric dow.



In [19]:
rollback_before_flag = True
rollback_after_flag = True

query = """


select s.city as store, 
       extract(dow from sa.sale_date) as dow,
       to_char(sa.sale_date, 'Day') as day_of_week, 
       p.description as product, 
       sum(quantity) as total_number_sold
from stores as s 
     join sales as sa 
         on s.store_id = sa.store_id
     join line_items as l
         on sa.store_id = l.store_id and sa.sale_id = l.sale_id
     join products as p
         on l.product_id = p.product_id
group by store, dow, day_of_week, product
order by store, dow, product

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store,dow,day_of_week,product,total_number_sold
0,Berkeley,0,Sunday,Brocolli Stir Fry,43560
1,Berkeley,0,Sunday,Chicken Salad,10983
2,Berkeley,0,Sunday,Curry Chicken,64634
3,Berkeley,0,Sunday,Eggplant Lasagna,76396
4,Berkeley,0,Sunday,Pistachio Salmon,86932
...,...,...,...,...,...
275,Seattle,6,Saturday,Eggplant Lasagna,70030
276,Seattle,6,Saturday,Pistachio Salmon,80192
277,Seattle,6,Saturday,Spinach Orzo,20112
278,Seattle,6,Saturday,Teriyaki Chicken,50691


## You try it - Create a view called v_store_sales_by_day_of_week that finds each store's sales by day of week. Display the store id, city, day of week in numeric for sorting purposes, day of week in string for display purposes, and total_sales. Test the view by selecting from it.  You can drop the view when you are done with it if you want.

In [20]:
connection.rollback()

query = """

drop view if exists v_store_sales_by_day_of_week;

create view v_store_sales_by_day_of_week
as
select s.store_id,
       s.city,
       extract(dow from sa.sale_date) as dow,
       to_char(sa.sale_date, 'Day') as day_of_week,
       sum(sa.total_amount) as total_sales
from stores s
     join sales as sa
        on s.store_id = sa.store_id
group by s.store_id, s.city, dow, day_of_week
;

"""

cursor.execute(query)

connection.commit()

In [21]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from v_store_sales_by_day_of_week
order by city, dow
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,store_id,city,dow,day_of_week,total_sales
0,1,Berkeley,0,Sunday,4694640
1,1,Berkeley,1,Monday,3340116
2,1,Berkeley,2,Tuesday,1752036
3,1,Berkeley,3,Wednesday,3546144
4,1,Berkeley,4,Thursday,3507660
5,1,Berkeley,5,Friday,3273240
6,1,Berkeley,6,Saturday,4927224
7,3,Dallas,0,Sunday,3650748
8,3,Dallas,1,Monday,2602980
9,3,Dallas,2,Tuesday,1352760


In [22]:
connection.rollback()

query = """

drop view if exists v_store_sales_by_day_of_week;

"""

cursor.execute(query)

connection.commit()

## You try it - using a type 2 subquery: find customers in Alameda, CA whose total sales is greater than the average sales for their zip code.  Display the customer_id, first_name, last_name, and total_sales. (this may take a while to run)

In [23]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select cu.customer_id, 
       cu.first_name, 
       cu.last_name,
       sum(sa.total_amount) as total_sales
from customers cu
     join sales as sa
         on cu.customer_id = sa.customer_id
where cu.city = 'Alameda' and cu.state = 'CA'
group by cu.customer_id, cu.first_name, cu.last_name
having sum(sa.total_amount) >
        (select avg(sa2.total_amount)
         from customers as cu2
              join sales as sa2
                  on cu2.customer_id = sa2.customer_id
         where cu.zip = cu2.zip)
;


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,customer_id,first_name,last_name,total_sales
0,3449,Whitby,Mennithorp,2076
1,3450,Royall,Kaas,4224
2,3451,Sula,Kneller,2784
3,3452,Casper,Burford,4260
4,3453,Leighton,Huxley,3408
...,...,...,...,...
298,4552,Krisha,Hovenden,2652
299,4553,Flori,Marchington,3144
300,4554,Rock,Boundey,3312
301,4555,Nana,Bartholin,3108
